In [16]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns

import matplotlib.pyplot as plt

# Load sample fraud scoring dataset
# Creating a synthetic dataset for AML/Financial Crime fraud scoring
np.random.seed(42)
n_samples = 1000

fraud_data = {
    'transaction_amount': np.random.uniform(10, 10000, n_samples),
    'customer_age': np.random.uniform(18, 80, n_samples),
    'account_age_days': np.random.uniform(1, 3650, n_samples),
    'num_transactions_30days': np.random.uniform(1, 100, n_samples),
    'failed_login_attempts': np.random.uniform(0, 10, n_samples),
    'international_transaction': np.random.randint(0, 2, n_samples),
    'fraud_score': np.random.uniform(0, 100, n_samples)  # Target variable
}

df = pd.DataFrame(fraud_data)
print("Fraud Scoring Dataset:")
df.head()
print(f"\nDataset shape: {df.shape}")

Fraud Scoring Dataset:

Dataset shape: (1000, 7)


In [17]:
import os

# Create the output folder if it doesn't exist
os.makedirs("data", exist_ok=True)

# Save dataframe to CSV and Excel
df.to_csv("data/fraud_scoring_dataset.csv", index=False)
df.to_excel("data/fraud_scoring_dataset.xlsx", index=False)

print("Files created:")
print("- data/fraud_scoring_dataset.csv")
print("- data/fraud_scoring_dataset.xlsx")

Files created:
- data/fraud_scoring_dataset.csv
- data/fraud_scoring_dataset.xlsx


In [18]:
df_loaded = pd.read_csv("data/fraud_scoring_dataset.csv")
df_loaded.head()

,transaction_amount,customer_age,account_age_days,num_transactions_30days,failed_login_attempts,international_transaction,fraud_score
0,3751.655787,29.478242,955.964040,67.597596,5.719959,0,51.874785
1,9507.635921,51.597859,902.225638,79.871458,8.054323,1,5.116551
2,7322.619479,72.122642,3307.922964,25.796322,7.601609,0,23.285856
3,5990.598257,63.397943,911.594083,62.862536,1.538999,1,49.200173
4,1568.626218,68.006791,993.344551,57.602852,1.492495,1,31.023528


In [19]:
# Add dummy names and country profile based on fraud_score, then overwrite CSV
np.random.seed(42)

first_names = [
    "Liam", "Olivia", "Noah", "Emma", "Ava", "Ethan", "Mia", "Lucas", "Sophia", "Amelia",
    "James", "Isabella", "Benjamin", "Charlotte", "Henry", "Harper", "Alexander", "Evelyn"
]
last_names = [
    "Smith", "Johnson", "Brown", "Taylor", "Anderson", "Thomas", "Jackson", "White", "Harris",
    "Martin", "Thompson", "Garcia", "Martinez", "Robinson", "Clark", "Lewis", "Walker", "Hall"
]

safe_countries = ["United States", "Canada", "Germany", "Netherlands", "Sweden", "Switzerland", "Norway", "Denmark", "Finland", "Ireland"]
medium_risk_countries = ["India", "Indonesia", "Philippines", "Vietnam", "Thailand", "Malaysia", "Turkey", "Brazil", "Mexico", "South Africa"]
high_risk_countries = ["Afghanistan", "Yemen", "Somalia", "Syria", "Libya", "Venezuela", "Iraq", "Sudan", "Myanmar", "North Korea"]

# Work on loaded CSV dataset
enriched_df = df_loaded.copy()

# Add dummy names
enriched_df["first_name"] = np.random.choice(first_names, size=len(enriched_df))
enriched_df["last_name"] = np.random.choice(last_names, size=len(enriched_df))

# Score bands: high score = safe countries, medium = medium-risk, low = high-risk
low_thr, high_thr = 33, 66
enriched_df["country"] = ""

low_mask = enriched_df["fraud_score"] < low_thr
mid_mask = (enriched_df["fraud_score"] >= low_thr) & (enriched_df["fraud_score"] < high_thr)
high_mask = enriched_df["fraud_score"] >= high_thr

enriched_df.loc[high_mask, "country"] = np.random.choice(safe_countries, size=high_mask.sum())
enriched_df.loc[mid_mask, "country"] = np.random.choice(medium_risk_countries, size=mid_mask.sum())
enriched_df.loc[low_mask, "country"] = np.random.choice(high_risk_countries, size=low_mask.sum())

# Optional label for clarity
enriched_df["country_risk_group"] = np.where(
    high_mask, "safe_country_group",
    np.where(mid_mask, "medium_risk_country_group", "high_risk_country_group")
)

# Save back to CSV
enriched_df.to_csv("data/fraud_scoring_dataset.csv", index=False)

print(enriched_df[["first_name", "last_name", "country", "country_risk_group", "fraud_score"]].head())
print("Updated CSV saved to: data/fraud_scoring_dataset.csv")

  first_name last_name       country         country_risk_group  fraud_score
0        Mia  Martinez  South Africa  medium_risk_country_group    51.874785
1      Henry   Jackson         Libya    high_risk_country_group     5.116551
2      James    Harris       Somalia    high_risk_country_group    23.285856
3      Lucas  Thompson        Mexico  medium_risk_country_group    49.200173
4        Mia  Robinson          Iraq    high_risk_country_group    31.023528
Updated CSV saved to: data/fraud_scoring_dataset.csv


In [21]:
enriched_df = pd.read_csv("data/fraud_scoring_dataset.csv")
enriched_df.head()

,transaction_amount,customer_age,account_age_days,num_transactions_30days,failed_login_attempts,international_transaction,fraud_score,first_name,last_name,country,country_risk_group
0,3751.655787,29.478242,955.964040,67.597596,5.719959,0,51.874785,Mia,Martinez,South Africa,medium_risk_country_group
1,9507.635921,51.597859,902.225638,79.871458,8.054323,1,5.116551,Henry,Jackson,Libya,high_risk_country_group
2,7322.619479,72.122642,3307.922964,25.796322,7.601609,0,23.285856,James,Harris,Somalia,high_risk_country_group
3,5990.598257,63.397943,911.594083,62.862536,1.538999,1,49.200173,Lucas,Thompson,Mexico,medium_risk_country_group
4,1568.626218,68.006791,993.344551,57.602852,1.492495,1,31.023528,Mia,Robinson,Iraq,high_risk_country_group


In [14]:
# Define sanctioned countries list
sanctioned_countries = ['Iran', 'North Korea', 'Russia', 'Syria', 'Cuba', 'Venezuela', 'Zimbabwe']

# Update the enriched_df with sanctioned country logic
enriched_df['is_sanctioned_country'] = enriched_df['country'].isin(sanctioned_countries)

# Update country_risk_group to reflect sanctioned status
enriched_df['country_risk_group'] = enriched_df['country_risk_group'].apply(
    lambda x: 'sanctioned_country_group' if enriched_df.loc[enriched_df['country_risk_group'] == x, 'is_sanctioned_country'].any() else x
)

# Better approach: reassign based on sanctioned status
enriched_df['country_risk_group'] = enriched_df.apply(
    lambda row: 'sanctioned_country_group' if row['is_sanctioned_country'] else row['country_risk_group'],
    axis=1
)

# Save updated CSV
enriched_df.to_csv("data/fraud_scoring_dataset.csv", index=False)

print("Updated dataset with sanctioned countries:")
print(enriched_df[['first_name', 'last_name', 'country', 'country_risk_group', 'is_sanctioned_country', 'fraud_score']].head(10))
print(f"\nTotal sanctioned country records: {enriched_df['is_sanctioned_country'].sum()}")
print("CSV updated and saved to: data/fraud_scoring_dataset.csv")

Updated dataset with sanctioned countries:
  first_name last_name       country         country_risk_group  \
0        Mia  Martinez  South Africa  medium_risk_country_group   
1      Henry   Jackson         Libya   sanctioned_country_group   
2      James    Harris       Somalia   sanctioned_country_group   
3      Lucas  Thompson        Mexico  medium_risk_country_group   
4        Mia  Robinson          Iraq   sanctioned_country_group   
5      James    Taylor          Iraq   sanctioned_country_group   
6      James  Martinez        Turkey  medium_risk_country_group   
7       Emma   Johnson       Myanmar   sanctioned_country_group   
8      Lucas  Anderson          Iraq   sanctioned_country_group   
9       Noah  Thompson      Malaysia  medium_risk_country_group   

   is_sanctioned_country  fraud_score  
0                  False    51.874785  
1                  False     5.116551  
2                  False    23.285856  
3                  False    49.200173  
4                  

In [15]:
# Update sanctioned countries list based on FATF standards
sanctioned_countries = ['Iran', 'North Korea', 'Russia', 'Syria', 'Cuba']

# Update the enriched_df with corrected sanctioned country logic
enriched_df['is_sanctioned_country'] = enriched_df['country'].isin(sanctioned_countries)

# Reassign country_risk_group based on sanctioned status and fraud score
enriched_df['country_risk_group'] = enriched_df.apply(
    lambda row: 'sanctioned_country_group' if row['is_sanctioned_country'] 
    else ('high_risk_country_group' if row['fraud_score'] < low_thr 
          else ('medium_risk_country_group' if row['fraud_score'] < high_thr 
                else 'safe_country_group')),
    axis=1
)

# Save updated CSV
enriched_df.to_csv("data/fraud_scoring_dataset.csv", index=False)

print("Updated dataset with corrected sanctioned countries:")
print(enriched_df[['first_name', 'last_name', 'country', 'country_risk_group', 'is_sanctioned_country', 'fraud_score']].head(15))
print(f"\nTotal sanctioned country records: {enriched_df['is_sanctioned_country'].sum()}")
print(f"High risk (non-sanctioned) records: {((enriched_df['fraud_score'] < low_thr) & (~enriched_df['is_sanctioned_country'])).sum()}")
print("CSV updated and saved to: data/fraud_scoring_dataset.csv")

Updated dataset with corrected sanctioned countries:
   first_name last_name       country         country_risk_group  \
0         Mia  Martinez  South Africa  medium_risk_country_group   
1       Henry   Jackson         Libya    high_risk_country_group   
2       James    Harris       Somalia    high_risk_country_group   
3       Lucas  Thompson        Mexico  medium_risk_country_group   
4         Mia  Robinson          Iraq    high_risk_country_group   
5       James    Taylor          Iraq    high_risk_country_group   
6       James  Martinez        Turkey  medium_risk_country_group   
7        Emma   Johnson       Myanmar    high_risk_country_group   
8       Lucas  Anderson          Iraq    high_risk_country_group   
9        Noah  Thompson      Malaysia  medium_risk_country_group   
10     Olivia    Thomas       Somalia    high_risk_country_group   
11   Isabella      Hall        Norway         safe_country_group   
12      Ethan    Martin       Denmark         safe_country_grou